# PRODUCTION

In [1]:
# ==== IMPORTS ====
import pandas as pd
import json
import re

# ==== SETTINGS ====
pd.set_option("display.max_colwidth", None)  # Show full text in cells

In [2]:
# ==== FUNCTIONS ====
# _________________________________________________________
# ---- Functions for loading and rinsing dataframes ----
def load_file(path):
    try:
        df = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path, sep=";", encoding="utf-8")
        resolved_path = path
    except FileNotFoundError:
        alt_path = f"../{path}"
        df = pd.read_parquet(alt_path) if alt_path.endswith(".parquet") else pd.read_csv(alt_path, sep=";", encoding="utf-8")
        resolved_path = alt_path
    return df, resolved_path

def load_json(path):
    try:
        with open(path, "r") as f:
            data = json.load(f)
        resolved_path = path
    except FileNotFoundError:
        alt_path = f"../{path}"
        with open(alt_path, "r") as f:
            data = json.load(f)
        resolved_path = alt_path
    return data, resolved_path

def df_info(df, path):
    print(f"DataFrame loaded from: {path}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}\n")

def duplicate_rinse(df,collumn):
    dup_mask = df[collumn].duplicated(keep=False)

    print(f"Rows with duplicated {collumn}: {dup_mask.sum()}")
    #display(df.loc[dup_mask].sort_values(by={collumn}))

    # Remove all rows where member_id_ss appears more than once
    df_rinsed = df.loc[~dup_mask].reset_index(drop=True)

    print(f"Shape after removing duplicated {collumn} rows: {df_rinsed.shape}\n")
    return df_rinsed

def run_info_n_rinse(df, path, collumn):
    name = path.split("/")[-1].rsplit(".", 1)[0]
    print(f"===== {name} =====")
    df_info(df, path)
    df_rinsed = duplicate_rinse(df, collumn)
    return df_rinsed
# ________________________________________________________

# ________________________________________________________
# ---- Functions for department mapping ----
def _flatten_text(value):
    """Return a flat list of strings from nested str/list/dict structures."""
    out = []
    if value is None:
        return out
    if isinstance(value, str):
        out.extend([v.strip() for v in value.split("|") if v.strip()])
    elif isinstance(value, list):
        for v in value:
            out.extend(_flatten_text(v))
    elif isinstance(value, dict):
        for v in value.values():
            out.extend(_flatten_text(v))
    return out

def _norm(text):
    text = str(text).strip().lower()
    text = re.sub(r"^dtu\s+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def _department_value(dep_item):
    dept = dep_item.get("department")
    if isinstance(dept, dict):
        return dept.get("en") or dept.get("da") or next((str(v) for v in dept.values() if v), None)
    return dept

def build_alias_lookup(dep_items):
    """Build alias -> department lookup from title + sections in dep JSON."""
    alias_lookup = {}
    for item in dep_items:
        department_val = _department_value(item)
        aliases = []
        aliases.extend(_flatten_text(item.get("title")))
        aliases.extend(_flatten_text(item.get("sections")))
        for alias in aliases:
            alias_lookup[_norm(alias)] = department_val
    return alias_lookup

def _match_department_from_text(raw_text, alias_lookup):
    """Try exact match first, then contains match on one text value."""
    if pd.isna(raw_text):
        return pd.NA

    txt_norm = _norm(raw_text)

    if txt_norm in alias_lookup:
        return alias_lookup[txt_norm]

    for alias, dep_val in alias_lookup.items():
        if alias in txt_norm or txt_norm in alias:
            return dep_val

    return pd.NA

def _match_from_affiliations(affiliations, alias_lookup):
    """
    Try matching each affiliations segment from left to right.
    Example format: "segment1|segment2|segment3"
    """
    if pd.isna(affiliations):
        return pd.NA

    parts = [part.strip() for part in str(affiliations).split("|") if part.strip()]
    for part in parts:
        match = _match_department_from_text(part, alias_lookup)
        if pd.notna(match):
            return match

    return pd.NA

def add_department_mapping(
    df,
    dep_items,
    publisher_col="Publisher",
    affiliations_col="Affiliations",
    output_col="Department_new",
):
    """
    Reusable mapping for any dataframe with publisher/affiliations columns.

    Strategy:
    1) Try matching department from publisher.
    2) If unmatched, fallback to affiliations segments split by '|', left -> right.
    """
    required_cols = [publisher_col, affiliations_col]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    alias_lookup = build_alias_lookup(dep_items)
    out_df = df.copy()

    out_df[output_col] = out_df[publisher_col].apply(
        lambda value: _match_department_from_text(value, alias_lookup)
    )

    unmatched_mask = out_df[output_col].isna()
    if unmatched_mask.any():
        out_df.loc[unmatched_mask, output_col] = out_df.loc[unmatched_mask, affiliations_col].apply(
            lambda value: _match_from_affiliations(value, alias_lookup)
        )

    return out_df

def dep_map_checks(df):
    """Department mapping checks"""
    matched_count = df["Department_new"].notna().sum()
    print("Matched rows:", matched_count, "/", len(df))
    #display(df[["Affiliations", "Publisher", "Department_new"]].head(20))

    ### Test for unmatched rows
    if df["Department_new"].isna().any():
        print("Not all rows were matched, the unmatched rows are:")
        print()

    unmatched_publishers = (
        df.loc[df["Department_new"].isna(), "Publisher"]
        .fillna("Missing")
        .astype(str)
        .str.strip()
        .replace("", "Missing")
        .value_counts()
        .rename_axis("Publisher")
        .reset_index(name="Count")
    )

    print(f"Unmatched rows: {unmatched_publishers['Count'].sum()} / {len(df)}")
    display(unmatched_publishers.head(30))
# ________________________________________________________

# ________________________________________________________
# ---- Export function(s) ----
def export_csv(df, path, name):
    """USAGE:

     export_csv(
    df=df_meta_rinsed,
    path="../[input your path here]/",
    name="[name of file here].csv",
    )
 """

    path_name = path + name
    df.to_csv(path_name, index=False, sep=";", encoding="utf-8")
    print(f"Exported updated DataFrame to: {path_name}")

In [3]:
# ==== MAIN ===

# Load and process metadata and metrics files
df_meta, path_meta = load_file("Data/gcp_order/dtu_findit/master_thesis_meta/thesis_meta_combined.parquet")
df_metrics, path_metrics = load_file("Data/extracted_metrics_reorder.csv")

# rinse duplicates based on member_id_ss and member_id_ss_metrics
df_meta_rinsed = run_info_n_rinse(df_meta, path_meta, "member_id_ss")
df_metrics_rinsed = run_info_n_rinse(df_metrics, path_metrics, "member_id_ss_metrics")



===== thesis_meta_combined =====
DataFrame loaded from: ../Data/gcp_order/dtu_findit/master_thesis_meta/thesis_meta_combined.parquet
Shape: (19690, 40)
Columns: ['abstract_ts', 'access_ss', 'Affiliations', 'Timestamp', 'Author', 'citation_count_i', 'ID', 'dtu_library_collection_facet', 'collection_facet', 'Publication Year', 'Conference', 'DOI', 'Editor', 'embargo_ssf', 'format', 'fulltext_availability_facet', 'has_openaccess_fulltext_b', 'holdings_ssf', 'ISBN', 'Journal Issue', 'journal_issue_tsort', 'journal_oa_model_ss', 'Journal Page', 'journal_page_start_tsort', 'Journal Title', 'journal_title_facet', 'toc_key_s', 'Journal Volume', 'journal_vol_tsort', 'keywords_ts', 'keywords_facet', 'keywords_normalized', 'isolanguage_facet', 'member_id_ss', 'ORCID', 'primary_member_id_s', 'Publisher', 'Source', 'source_all_ss', 'Title']

Rows with duplicated member_id_ss: 11368
Shape after removing duplicated member_id_ss rows: (8322, 40)

===== extracted_metrics_reorder =====
DataFrame loaded 

In [4]:
# Load helper file: the JSON file with the departments and sections of DTU
dep, dep_path = load_json("../Data/gcp_order/helper_files/department_classification.json")
print(f"Loaded department mapping from: {dep_path}")

# Add new column with department mapping based on publisher and affiliations
df_meta_rinsed = add_department_mapping(
    df=df_meta_rinsed,
    dep_items=dep,
    publisher_col="Publisher",
    affiliations_col="Affiliations",
    output_col="Department_new",
)
# Department mapping checks
dep_map_checks(df_meta_rinsed)

df_meta_rinsed = run_info_n_rinse(df_meta_rinsed, path_meta, "member_id_ss")

Loaded department mapping from: ../Data/gcp_order/helper_files/department_classification.json
Matched rows: 8204 / 8322
Not all rows were matched, the unmatched rows are:

Unmatched rows: 118 / 8322


,Publisher,Count
0,Missing,115
1,Technical University of Denmark (DTU),3


===== thesis_meta_combined =====
DataFrame loaded from: ../Data/gcp_order/dtu_findit/master_thesis_meta/thesis_meta_combined.parquet
Shape: (8322, 41)
Columns: ['abstract_ts', 'access_ss', 'Affiliations', 'Timestamp', 'Author', 'citation_count_i', 'ID', 'dtu_library_collection_facet', 'collection_facet', 'Publication Year', 'Conference', 'DOI', 'Editor', 'embargo_ssf', 'format', 'fulltext_availability_facet', 'has_openaccess_fulltext_b', 'holdings_ssf', 'ISBN', 'Journal Issue', 'journal_issue_tsort', 'journal_oa_model_ss', 'Journal Page', 'journal_page_start_tsort', 'Journal Title', 'journal_title_facet', 'toc_key_s', 'Journal Volume', 'journal_vol_tsort', 'keywords_ts', 'keywords_facet', 'keywords_normalized', 'isolanguage_facet', 'member_id_ss', 'ORCID', 'primary_member_id_s', 'Publisher', 'Source', 'source_all_ss', 'Title', 'Department_new']

Rows with duplicated member_id_ss: 0
Shape after removing duplicated member_id_ss rows: (8322, 41)



# DEVELOPMENT

## TO DO: 

- [x] get pdf page count (incl/excl. appendix) \check
- [x] get word count \check
- [x] rewrite for GCP and append results to csv file
- [] populate df metrics_analysis with data from metrics and meta that is relevant for analysis
- [] explore the dataset 'Data/extracted_metrics.csv'
- [] get supervisor(s)

In [ ]:
# ==== FIND & PRINT DUPLICATES (COLUMN: tbd) ====
dup_mask = df_csv.duplicated(keep=False)
#print(df_csv[dup_mask].sort_values(by=list(df_csv.columns)))

# ==== REMOVE DUPLICATES ====
#df_csv_unique = df_csv.drop_duplicates().reset_index(drop=True)

In [ ]:


# ==== FIND & PRINT DUPLICATES ====
dup_mask = df_csv.duplicated(keep=False)
#print(df_csv[dup_mask].sort_values(by=list(df_csv.columns)))

# ==== REMOVE DUPLICATES ====
df_csv_unique = df_csv.drop_duplicates().reset_index(drop=True)

print(f"Unique DataFrame has shape: {df_csv_unique.shape}")  # Print the shape of the unique DataFrame to check if duplicates were removed correctly

#display(df_csv.head(3))  # Display the first few rows of the DataFrame

# Archives - old stuff

In [ ]:
test_file = "../Data/RAW_test/5d1c8d79d9001d146569a4dc_Deep speech recognition in Danish (translated Dyb talegenkendelse pa dansk).pdf"
test_file = "../Data/RAW_test/5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf"
with open(test_file, 'rb') as file:
    try:
        reader = pypdf.PdfReader(file)
    except Exception as e:
        print(f"Error reading PDF file: {e}")

    # display page number
    number = 51
    page = reader.pages[number]
    text = page.extract_text()
    print(f"Page {number}:\n{text}\n{'-' * 40}")

In [ ]:
df_organized = df_csv_metrics_rinsed[['pdf_file', 'member_id_ss_metrics', 'num_tot_pages', 'num_cont_pages', 'match_trigger', 'num_words_full', 'num_words_cont']]
print(df_organized.columns)
print(df_organized.shape)

display(df_organized.head(3))  # Display the first few rows of the organized DataFrame

output_path = csv_path2.replace("extracted_metrics.csv", "extracted_metrics_reorder.csv")
df_organized.to_csv(output_path, sep=";", encoding="utf-8", index=False)
print(f"Saved to {output_path}")